# Two Buildings PACER Example

## Setup
1. [PACER stack deployed](https://github.com/NatLabRockies/alfalfa/wiki/Deployment) locally with **at least two workers**

## Notes
1. For PACER v0.1.0, API reads and writes are implemented over the [Haystack API](https://project-haystack.org/doc/docHaystack/HttpApi), particularly the `read` and `pointWrite` endpoints.


In [ ]:
import datetime
from pacer_client.pacer_client import PacerClient
from pprint import pprint

### Create new PACER client object
If you are not hosting the PACER server yourself replace the `host` with the one of your server without a trailing slash (this is a known bug and will eventually be fixed)

In [ ]:
ac = PacerClient(host='http://localhost')

### Define paths to models to be uploaded

In [ ]:
model_paths = ['./twobldgs','./twobldgs']

### Upload sites to PACER
The `ac.submit` function returns the site_id which is used to interact with that specific site over the API. A simulation is a site.


Sites can be viewed at http://localhost/sites

Metadata generated can be viewed at http://localhost/api/read?filter=point

In [ ]:
site_ids = ac.submit(model_paths)


In [ ]:
print(site_ids)

### Define parameters to run the simulations

In [ ]:
# If you are using historian, you will need to search for this time period in Grafana dashboard to view results.
start_dt = datetime.datetime(2021, 7, 1, 12, 2, 0)
end_dt = datetime.datetime(2021, 7, 3, 0, 0, 0)

# For external_clock == true, API calls are used to advance the model.
# If external_clock == false, PACER will handle advancing the model according to a specified timescale (timescale 1 => realtime)
params = {
    "external_clock": True,
    "start_datetime": start_dt,
    "end_datetime": end_dt
}

## Start simulations 
Note: one sim runs / worker, so if you have not scaled your local deployment to workers >= 2, the second worker won't have a chance to start and this code block will not complete.

In [ ]:
print(f"Starting sites: {site_ids}")
ac.start(site_ids, **params) # If the code is stuck here you may not have enough workers for 2 models at the same time

### Get the model's input points
Get a list of all of the model's input points and their values


In [ ]:
print(f"{site_ids[0]} inputs:")
pprint(ac.get_inputs(site_ids[0]))
print(f"{site_ids[1]} inputs:")
pprint(ac.get_inputs(site_ids[1]))

## Set model input point

To set an input value use the `ac.set_inputs(site_id, inputs)` function. 
- `site_id` - the id of the site returned by the `ac.submit` function
- `inputs` - a dictionary of input names and the desired values


In [ ]:
input_dict = {'OfficeSmall HTGSETP_SCH_NO_OPTIMUM': 20}
ac.set_inputs(site_ids[0], input_dict)

### Advance the model
12/10/2021: timestep is hardcoded to 1 minute w/in PACER worker.
Model values are exposed as the `curVal` for points and can be viewed at `http://localhost/api/read?filter=point`

In [ ]:
timesteps = 5
for _ in range(timesteps):
    ac.advance(site_ids)
    print(f"Model advanced to time: {ac.get_sim_time(site_ids[0])}")

### Get model's outputs
Query the outputs of the models as well as their values

In [ ]:
print(f"{site_ids[0]} outputs:")
pprint(ac.get_outputs(site_ids[0]))
print(f"{site_ids[1]} outputs:")
pprint(ac.get_outputs(site_ids[1]))

### Stop the simulations

In [ ]:
ac.stop(site_ids)